In [20]:
import stanza
import pandas as pd
import re
from IPython.display import Markdown, display

def show_md(txt):
    display(Markdown(txt))

stanza.download("te", verbose=False)

nlp = stanza.Pipeline("te", processors="tokenize,pos,lemma,depparse", use_gpu=False, verbose=False)

print("Stanza Telugu pipeline set up")


Stanza Telugu pipeline set up


In [5]:
def pos_tag_sentence(text, show_table=True):
    doc = nlp(text)
    rows = []
    for si, sent in enumerate(doc.sentences, start=1):
        for wi, w in enumerate(sent.words, start=1):
            rows.append({
                "sent_id": si,
                "word_index": wi,
                "text": w.text,
                "upos": w.upos,
                "xpos": w.xpos,
                "feats": w.feats,
                "deprel": w.deprel,
                "head": w.head
            })
    df = pd.DataFrame(rows)
    if show_table:
        display(df)
    return doc, df


In [6]:
# Pronouns
PRONOUN_FEATURES = {
    "నేను": {"Person": "1", "Number": "Sing"},
    "నువ్వు": {"Person": "2", "Number": "Sing"},
    "మీరు": {"Person": "2", "Number": "Plur"},
    "మేము": {"Person": "1", "Number": "Plur"},
    "అతను": {"Person": "3", "Number": "Sing", "Gender": "Masc"},
    "ఆమె": {"Person": "3", "Number": "Sing", "Gender": "Fem"},
    "వారు": {"Person": "3", "Number": "Plur"},
    "వాళ్లు": {"Person": "3", "Number": "Plur"}
}

# verb suffix patterns
VERB_SUFFIXES = {
    "ను": {"Person": "1", "Number": "Sing"},
    "ము": {"Person": "1", "Number": "Plur"},
    "వు": {"Person": "2", "Number": "Sing"},
    "రు": {"Person": "2", "Number": "Plur"},
    "డు": {"Person": "3", "Number": "Sing", "Gender": "Masc"},
    "ారు": {"Person": "3", "Number": "Plur"},
    "న్నాను": {"Person": "1", "Number": "Sing"},
    "న్నాము": {"Person": "1", "Number": "Plur"},
    "న్నాడు": {"Person": "3", "Number": "Sing", "Gender": "Masc"},
    "న్నారు": {"Person": "3", "Number": "Plur"},
    "ుతున్నాను": {"Tense": "Pres", "Aspect": "Prog", "Person": "1", "Number": "Sing"},
    "ుతున్నాడు": {"Tense": "Pres", "Aspect": "Prog", "Person": "3", "Number": "Sing"},
    "ుతున్నారని": {},
}

# Case-postposition heuristics
CASE_MARKERS = {
    "ను": "Acc",
    "కి": "Dat",
    "కు": "Dat",
    "తో": "Instr",
    "పై": "Loc",
    "లో": "Loc",
    "పైకి": "Loc",
    " నుండి": "Abl",
}

def parse_feats_from_stanza(word):
    feats = {"Person": None, "Number": None, "Gender": None, "Tense": None}
    if word.feats:
        parts = word.feats.split("|")
        for p in parts:
            if "=" in p:
                k, v = p.split("=")
                if k in feats:
                    feats[k] = v
    return feats

def infer_features(word):
    text = word.text.strip()
    feats = parse_feats_from_stanza(word)

    if text in PRONOUN_FEATURES:
        for k, v in PRONOUN_FEATURES[text].items():
            feats[k] = v

    if word.upos == "VERB" or word.upos == "AUX" or any(text.endswith(suf) for suf in VERB_SUFFIXES):
        for suf in sorted(VERB_SUFFIXES.keys(), key=lambda x: -len(x)):
            if text.endswith(suf):
                for k, v in VERB_SUFFIXES[suf].items():
                    feats[k] = v
                break

    if word.upos in ("NOUN", "PRON", "PROPN"):
        for suf in sorted(CASE_MARKERS.keys(), key=lambda x: -len(x)):
            if text.endswith(suf.strip()):
                feats["Case"] = CASE_MARKERS[suf]
                break

    return feats


In [7]:
def analyze_and_print(text):
    doc, df = pos_tag_sentence(text)
    sent = doc.sentences[0]
    print("\n-- Token features (stanza + inferred) --")
    for w in sent.words:
        inferred = infer_features(w)
        feats_str = w.feats if w.feats is not None else ""
        print(f"{w.text:12s} | UPOS={w.upos:6s} | FEATS(stanza)={feats_str:20s} | INFER={inferred}")
    return doc

In [8]:
tests = [
    "నేను ఆపిల్ తిన్నాను",         # correct
    "నేను ఆపిల్ తిన్నాము",        # mismatch- single subject plural verb
    "మేము ఆపిల్ తిన్నాము",       # correct
    "వారు వస్తున్నారు",           # correct
    "ఆమె తిన్నాడు",             # gender mismatch
    "నేను బుక్ కు వెళ్ళాను",      # correct dates
    "నువ్వు వస్తున్నావు",          # correct
    "ఆపిల్ ను తినాను",           # correct
    "నేను తినలేదు లేదు"           # double negative
]

for s in tests:
    print("\n==============================")
    show_md(f"### Sentence: `{s}`")
    analyze_and_print(s)


### Sentence: `నేను ఆపిల్ తిన్నాను`

,sent_id,word_index,text,upos,xpos,feats,deprel,head
0,1,1,నేను,PRON,PRON,None,nsubj,3
1,1,2,ఆపిల్,NOUN,NOUN,None,compound:lvc,3
2,1,3,తిన్నాను,VERB,VERB,None,root,0



-- Token features (stanza + inferred) --
నేను         | UPOS=PRON   | FEATS(stanza)=                     | INFER={'Person': '1', 'Number': 'Sing', 'Gender': None, 'Tense': None, 'Case': 'Acc'}
ఆపిల్        | UPOS=NOUN   | FEATS(stanza)=                     | INFER={'Person': None, 'Number': None, 'Gender': None, 'Tense': None}
తిన్నాను     | UPOS=VERB   | FEATS(stanza)=                     | INFER={'Person': '1', 'Number': 'Sing', 'Gender': None, 'Tense': None}



### Sentence: `నేను ఆపిల్ తిన్నాము`

,sent_id,word_index,text,upos,xpos,feats,deprel,head
0,1,1,నేను,PRON,PRON,None,nsubj,3
1,1,2,ఆపిల్,NOUN,NOUN,None,obj,3
2,1,3,తిన్నాము,VERB,VERB,None,root,0


...

,sent_id,word_index,text,upos,xpos,feats,deprel,head
0,1,1,ఆపిల్,PRON,PRON,None,obj,3
1,1,2,ను,ADP,ADP,None,case,1
2,1,3,తినాను,VERB,VERB,None,root,0



-- Token features (stanza + inferred) --
ఆపిల్        | UPOS=PRON   | FEATS(stanza)=                     | INFER={'Person': None, 'Number': None, 'Gender': None, 'Tense': None}
ను           | UPOS=ADP    | FEATS(stanza)=                     | INFER={'Person': '1', 'Number': 'Sing', 'Gender': None, 'Tense': None}
తినాను       | UPOS=VERB   | FEATS(stanza)=                     | INFER={'Person': '1', 'Number': 'Sing', 'Gender': None, 'Tense': None}



### Sentence: `నేను తినలేదు లేదు`

,sent_id,word_index,text,upos,xpos,feats,deprel,head
0,1,1,నేను,PRON,PRON,None,nsubj,3
1,1,2,తినలేదు,VERB,VERB,None,compound:svc,3
2,1,3,లేదు,VERB,VERB,None,root,0



-- Token features (stanza + inferred) --
నేను         | UPOS=PRON   | FEATS(stanza)=                     | INFER={'Person': '1', 'Number': 'Sing', 'Gender': None, 'Tense': None, 'Case': 'Acc'}
తినలేదు      | UPOS=VERB   | FEATS(stanza)=                     | INFER={'Person': None, 'Number': None, 'Gender': None, 'Tense': None}
లేదు         | UPOS=VERB   | FEATS(stanza)=                     | INFER={'Person': None, 'Number': None, 'Gender': None, 'Tense': None}


In [9]:
def get_subject_and_verb(sentence):
    subj = None
    verb = None
    for w in sentence.words:
        if w.deprel in ("nsubj", "nsubj:pass"):
            subj = w
        if w.upos == "VERB" and w.head == 0:
            verb = w
    return subj, verb


In [10]:
def check_subject_verb_agreement(sentence):
    subj, verb = get_subject_and_verb(sentence)
    if not subj or not verb:
        return []

    subj_feats = infer_features(subj)
    verb_feats = infer_features(verb)

    errors = []

    # person/number agreement
    if subj_feats.get("Person") and verb_feats.get("Person"):
        if subj_feats["Person"] != verb_feats["Person"]:
            errors.append(f"Subject–verb person mismatch ({subj.text} vs {verb.text})")

    if subj_feats.get("Number") and verb_feats.get("Number"):
        if subj_feats["Number"] != verb_feats["Number"]:
            errors.append(f"Subject–verb number mismatch ({subj.text} vs {verb.text})")

    # gender mismatch
    if subj_feats.get("Gender") and verb_feats.get("Gender"):
        if subj_feats["Gender"] != verb_feats["Gender"]:
            errors.append(f"Gender disagreement between {subj.text} and {verb.text}")

    return errors


In [11]:
def check_case_markers(sentence):
    errors = []
    for i, w in enumerate(sentence.words):
        if w.upos == "ADP" and i > 0:
            prev = sentence.words[i-1]
            if prev.upos not in ("NOUN", "PRON", "PROPN"):
                errors.append(f"Unexpected case marker '{w.text}' after {prev.text}")
    return errors


In [12]:
def check_tense_consistency(sentence):
    verb_feats = [infer_features(w) for w in sentence.words if w.upos == "VERB"]
    tenses = {f.get("Tense") for f in verb_feats if f.get("Tense")}
    if len(tenses) > 1:
        return [f"Inconsistent tenses used: {tenses}"]
    return []


In [22]:
def detect_negation(word):
    neg_patterns = [
        r".*లేదు$",    # ends with 'లేదు'
        r".*లేను$",    # ends with 'లేను'
        r".*వద్దు$",   # ends with 'వద్దు'
        r".*కాదు$",    # ends with 'కాదు'
        r".*లేక.*",    # contains 'లేక'
        r".*అక్కర్లేదు$",  # ends with 'అక్కర్లేదు'
    ]
    return any(re.match(pat, word) for pat in neg_patterns)


def check_double_negatives(sentence):
    neg_words = [w.text for w in sentence.words if detect_negation(w.text)]

    errors = []
    if len(neg_words) > 1:
        errors.append(f"Double negatives: {' '.join(neg_words)}")
    return errors


In [25]:
def grammar_check(text):
    doc = nlp(text)
    all_errors = []
    for sent in doc.sentences:
        errs = []
        errs += check_subject_verb_agreement(sent)
        errs += check_case_markers(sent)
        errs += check_tense_consistency(sent)
        errs += check_double_negatives(sent)

        if errs:
            show_md(f"**✖ Sentence:** `{text}`")
            for e in errs:
                print(" -", e)
            all_errors.extend(errs)
        else:
            show_md(f"✓ Sentence Correct: `{text}`")
    return all_errors


In [26]:
tests = [
    "నేను ఆపిల్ తిన్నాను",         # correct
    "నేను ఆపిల్ తిన్నాము",        # singular plural
    "మేము ఆపిల్ తిన్నాము",       # correct
    "వారు వస్తున్నారు",           # correct
    "ఆమె తిన్నాడు",             # gender mismatch
    "ఆపిల్ ను తినాను",           # correct
    "నేను తినలేదు లేదు"           # double negative
]

for t in tests:
    print("-----------------------------")
    grammar_check(t)


-----------------------------


✓ Sentence Correct: `నేను ఆపిల్ తిన్నాను`

-----------------------------


**✖ Sentence:** `నేను ఆపిల్ తిన్నాము`

 - Subject–verb number mismatch (నేను vs తిన్నాము)
-----------------------------


...

 - Gender disagreement between ఆమె and తిన్నాడు
-----------------------------


✓ Sentence Correct: `ఆపిల్ ను తినాను`

-----------------------------


**✖ Sentence:** `నేను తినలేదు లేదు`

 - Double negatives: తినలేదు లేదు
